<a href="https://colab.research.google.com/github/raheelarif86/AI_Training_November25/blob/main/OT_Risk_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The Project is destined for doing Risk Assessment in Operational Technology by using mutiple agents. This will bring super efficiency in conducting thorouguh Risk Assessments in a much smarter way while fulfilling even Adhoc Assessment needs in a much swifter way.

In [1]:
#install packages
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guida

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

We will be using OpenAI's `gpt-3.5-turbo`.



In [3]:
#setting the model
import os
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
#os.environ["OPENAI_MODEL_NAME"] = 'gpt-4.1-nano-2025-04-14'

Agents Creation:
1.

In [4]:
# Retrieve API key from Colab secrets and set it in environment variables
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('openai')

In [5]:
from crewai_tools import (
    DirectoryReadTool,
    FileReadTool,
    SerperDevTool,
    WebsiteSearchTool
)

In [6]:
# Set up API keys - enter from https://serper.dev/billing
os.environ["SERPER_API_KEY"] = "218873db9b73d345d3083e2f1f29cb74b14d4ddb" # serper.dev API key

# Instantiate tools
docs_tool = DirectoryReadTool(directory='./blog-posts')
file_tool = FileReadTool()
search_tool = SerperDevTool()
web_rag_tool = WebsiteSearchTool()

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic."    ,
    tools=[search_tool, web_rag_tool,docs_tool,file_tool],
    allow_delegation=False,
	verbose=True
)

In [1]:
!pip install gradio crewai pydantic

In [2]:
import gradio as gr
from crewai import Agent, Task, Crew

In [4]:
# Retrieve API key from Colab secrets and set it in environment variables
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('openai')

In [5]:
# Likelihood Calculator Agent
likelihood_agent = Agent(
    role="OT Cyber Likelihood Analyst",
    goal="Estimate the likelihood of a cyber incident in an OT environment",
    backstory=(
        "A senior OT security analyst specializing in threat modeling, "
        "attack surface analysis, and adversary capability assessment "
        "for industrial control systems."
    ),
    allow_delegation=False
)

# Impact Calculator Agent
impact_agent = Agent(
    role="OT Impact Assessment Specialist",
    goal="Evaluate operational, safety, and financial impact of cyber incidents",
    backstory=(
        "An expert in industrial safety, production continuity, and "
        "critical infrastructure resilience, experienced with IEC 62443 "
        "and NIST SP 800-82."
    ),
    allow_delegation=False
)

# Risk Estimator Agent
risk_agent = Agent(
    role="OT Risk Estimator",
    goal="Determine overall cyber risk based on likelihood and impact",
    backstory=(
        "A risk professional who applies quantitative and qualitative "
        "risk models to industrial environments and produces clear risk ratings."
    ),
    allow_delegation=False
)

# Cyber Countermeasure Planner Agent
countermeasure_agent = Agent(
    role="OT Cyber Countermeasure Planner",
    goal="Recommend OT-safe cybersecurity controls and mitigations",
    backstory=(
        "A control systems security architect with deep knowledge of "
        "IEC 62443, Purdue Model, and MITRE ATT&CK for ICS."
    ),
    allow_delegation=False
)

In [6]:
def run_ot_risk_assessment(
    asset,
    sector,
    threat_actor,
    exposure_level,
    controls_maturity,
    vulnerabilities
):
    # --- Task 1: Likelihood ---
    likelihood_task = Task(
        description=f"""
        Analyze the likelihood of a cyber incident for the following OT asset:

        Asset: {asset}
        Sector: {sector}
        Threat Actor: {threat_actor}
        Exposure Level: {exposure_level}
        Existing Controls Maturity: {controls_maturity}
        Vulnerabilities: {vulnerabilities}

        Consider attacker capability, exposure, and defensive maturity.
        Provide a likelihood score (Low/Medium/High) and reasoning.
        """,
        expected_output=(
            "Likelihood rating (Low/Medium/High), "
            "brief explanation, and contributing factors."
        ),
        agent=likelihood_agent
    )

    # --- Task 2: Impact ---
    impact_task = Task(
        description=f"""
        Assess the potential impact if a cyber attack succeeds against:

        Asset: {asset}
        Sector: {sector}

        Consider:
        - Safety (human life)
        - Production downtime
        - Equipment damage
        - Environmental impact
        - Regulatory and financial consequences

        Provide an impact rating and explanation.
        """,
        expected_output=(
            "Impact rating (Low/Medium/High/Critical), "
            "affected impact domains, and justification."
        ),
        agent=impact_agent
    )

    # --- Task 3: Risk Estimation ---
    risk_task = Task(
        description="""
        Using the likelihood and impact assessments,
        calculate the overall OT cyber risk.

        Use a risk matrix or qualitative formula.
        Provide:
        - Overall risk rating
        - Risk justification
        - Risk acceptability (Acceptable / Tolerable / Unacceptable)
        """,
        expected_output=(
            "Overall risk rating, risk score or matrix position, "
            "and acceptability decision."
        ),
        agent=risk_agent
    )

    # --- Task 4: Countermeasure Planning ---
    countermeasure_task = Task(
        description=f"""
        Based on the identified OT cyber risk,
        recommend OT-safe cybersecurity countermeasures.

        Align recommendations with:
        - IEC 62443
        - NIST SP 800-82
        - Purdue Model
        - MITRE ATT&CK for ICS

        Prioritize actions and avoid disrupting operations.
        """,
        expected_output=(
            "Prioritized list of cybersecurity controls with "
            "implementation guidance suitable for OT environments."
        ),
        agent=countermeasure_agent
    )

    # --- Setup Crew ---
    crew = Crew(
        agents=[
            likelihood_agent,
            impact_agent,
            risk_agent,
            countermeasure_agent
        ],
        tasks=[
            likelihood_task,
            impact_task,
            risk_task,
            countermeasure_task
        ],
        verbose=True
    )

    # --- Run ---
    result = crew.kickoff()
    return result

In [7]:
def assess_risk(
    asset,
    sector,
    threat_actor,
    exposure_level,
    controls_maturity,
    vulnerabilities
):
    return run_ot_risk_assessment(
        asset,
        sector,
        threat_actor,
        exposure_level,
        controls_maturity,
        vulnerabilities
    )


iface = gr.Interface(
    fn=assess_risk,
    inputs=[
        gr.Textbox(label="OT Asset", placeholder="PLC controlling turbine"),
        gr.Dropdown(
            ["Power", "Oil & Gas", "Manufacturing", "Water"],
            label="Sector"
        ),
        gr.Dropdown(
            ["Nation-State", "Cybercriminal", "Insider", "Hacktivist"],
            label="Threat Actor"
        ),
        gr.Dropdown(
            ["Low", "Medium", "High"],
            label="Exposure Level"
        ),
        gr.Dropdown(
            ["Weak", "Moderate", "Strong"],
            label="Existing Controls Maturity"
        ),
        gr.Textbox(
            label="Known Vulnerabilities",
            placeholder="Unpatched firmware, Flat network"
        )
    ],
    outputs=gr.Textbox(lines=25, label="OT Cyber Risk Assessment Report"),
    title="Agentic OT Cyber Risk Assessment",
    description=(
        "Multi-agent OT cyber risk assessment using CrewAI. "
        "Evaluates likelihood, impact, risk, and countermeasures "
        "aligned with IEC 62443 and NIST SP 800-82."
    )
)

if __name__ == "__main__":
    iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c790879571bf70a2fc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
# ================================
# INSTALL DEPENDENCIES
# ================================
!pip install -q gradio crewai

# ================================
# IMPORTS
# ================================
import gradio as gr
from crewai import Agent, Task, Crew

# ================================
# DEFINE NUCLEAR CYBER AGENTS
# ================================

likelihood_agent = Agent(
    role="Nuclear Cyber Threat Analyst",
    goal="Assess likelihood of cyber compromise in nuclear facilities",
    backstory=(
        "A nuclear cyber security specialist experienced in defense-in-depth, "
        "threat-informed risk assessment, insider threat analysis, and "
        "digital system isolation in accordance with NEI guidance and FANR regulations."
    ),
    allow_delegation=False
)

impact_agent = Agent(
    role="Nuclear Safety & Consequence Analyst",
    goal="Evaluate nuclear safety, radiological, and regulatory impact of cyber incidents",
    backstory=(
        "A nuclear safety engineer with deep knowledge of reactor protection systems, "
        "emergency preparedness, safety classification, and FANR licensing basis."
    ),
    allow_delegation=False
)

risk_agent = Agent(
    role="Nuclear Cyber Risk Authority",
    goal="Determine overall nuclear cyber risk significance and regulatory acceptability",
    backstory=(
        "A senior nuclear risk authority responsible for determining whether cyber risks "
        "are acceptable, conditionally acceptable, or unacceptable under FANR regulations."
    ),
    allow_delegation=False
)

countermeasure_agent = Agent(
    role="Nuclear Cyber Protection Architect",
    goal="Recommend NEI- and FANR-compliant cyber security controls",
    backstory=(
        "A nuclear cyber protection architect specializing in NEI cyber security plans, "
        "defense-in-depth, safety system isolation, monitoring, and regulatory compliance."
    ),
    allow_delegation=False
)

# ================================
# NUCLEAR RISK ASSESSMENT PIPELINE
# ================================

def run_nuclear_cyber_risk_assessment(
    facility_type,
    digital_asset,
    safety_class,
    threat_actor,
    network_connectivity,
    defensive_posture,
    vulnerabilities
):
    # --- Likelihood Task ---
    likelihood_task = Task(
        description=f"""
        Perform a cyber threat likelihood assessment for a nuclear facility.

        Facility Type: {facility_type}
        Digital Asset: {digital_asset}
        Safety Classification: {safety_class}
        Threat Actor: {threat_actor}
        Network Connectivity: {network_connectivity}
        Defensive Posture: {defensive_posture}
        Known Vulnerabilities: {vulnerabilities}

        Apply NEI defense-in-depth principles and nuclear threat modeling.
        """,
        expected_output=(
            "Likelihood rating (Low / Moderate / High) with clear justification "
            "and threat vectors relevant to nuclear environments."
        ),
        agent=likelihood_agent
    )

    # --- Impact Task ---
    impact_task = Task(
        description=f"""
        Assess the consequences of a successful cyber compromise of:

        Digital Asset: {digital_asset}
        Safety Classification: {safety_class}

        Consider:
        - Nuclear safety significance
        - Radiological consequences
        - Emergency preparedness impact
        - Physical security degradation
        - FANR regulatory non-compliance
        """,
        expected_output=(
            "Impact classification (Low / High / Critical) including affected "
            "safety functions and regulatory implications."
        ),
        agent=impact_agent
    )

    # --- Risk Task ---
    risk_task = Task(
        description="""
        Integrate likelihood and impact results to determine overall
        nuclear cyber risk significance.

        Determine:
        - Risk significance category
        - Impact on licensing basis
        - Acceptability under FANR regulations
        """,
        expected_output=(
            "Overall nuclear cyber risk classification "
            "(Acceptable / Conditionally Acceptable / Unacceptable) "
            "with regulatory justification."
        ),
        agent=risk_agent
    )

    # --- Countermeasure Task ---
    countermeasure_task = Task(
        description="""
        Define nuclear-safe cyber security countermeasures aligned with:

        - NEI cyber security expectations
        - FANR regulations
        - Defense-in-depth strategy
        - Safety system isolation requirements

        Prioritize controls without introducing operational risk.
        """,
        expected_output=(
            "Prioritized list of nuclear cyber security controls mapped to "
            "NEI and FANR requirements."
        ),
        agent=countermeasure_agent
    )

    # --- Crew Setup ---
    crew = Crew(
        agents=[
            likelihood_agent,
            impact_agent,
            risk_agent,
            countermeasure_agent
        ],
        tasks=[
            likelihood_task,
            impact_task,
            risk_task,
            countermeasure_task
        ],
        verbose=True
    )

    return crew.kickoff()

# ================================
# GRADIO INTERFACE
# ================================

def assess_nuclear_cyber_risk(
    facility_type,
    digital_asset,
    safety_class,
    threat_actor,
    network_connectivity,
    defensive_posture,
    vulnerabilities
):
    return run_nuclear_cyber_risk_assessment(
        facility_type,
        digital_asset,
        safety_class,
        threat_actor,
        network_connectivity,
        defensive_posture,
        vulnerabilities
    )


iface = gr.Interface(
    fn=assess_nuclear_cyber_risk,
    inputs=[
        gr.Dropdown(
            ["Nuclear Power Plant", "Research Reactor", "Fuel Handling Facility"],
            label="Facility Type"
        ),
        gr.Textbox(
            label="Digital Asset",
            placeholder="Reactor Protection System PLC"
        ),
        gr.Dropdown(
            ["Safety", "Security", "Emergency Preparedness", "Support"],
            label="System Safety Classification"
        ),
        gr.Dropdown(
            ["Nation-State", "Insider", "Terrorist Organization"],
            label="Threat Actor"
        ),
        gr.Dropdown(
            ["Isolated", "Controlled Interface", "Indirect Connectivity"],
            label="Network Connectivity"
        ),
        gr.Dropdown(
            ["Strong", "Moderate", "Weak"],
            label="Defensive Posture"
        ),
        gr.Textbox(
            label="Known Vulnerabilities",
            placeholder="Legacy firmware, shared maintenance laptop"
        )
    ],
    outputs=gr.Textbox(
        lines=30,
        label="Nuclear Cyber Security Risk Assessment Report"
    ),
    title="☢️ Agentic Nuclear Cyber Security Risk Assessment",
    description=(
        "CrewAI-powered nuclear cyber risk assessment aligned with "
        "NEI standards and FANR regulatory expectations. "
        "Designed for safety-critical nuclear environments."
    )
)

# ================================
# LAUNCH
# ================================
if __name__ == "__main__":
    iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1f4437897622f95113.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
# ======================================
# INSTALL DEPENDENCIES
# ======================================
!pip install -q gradio crewai langchain-openai

# ======================================
# IMPORTS
# ======================================
import os
import gradio as gr
from crewai import Agent, Task, Crew
from langchain_openai import ChatOpenAI

# ======================================
# OPENAI API KEY (SET YOUR KEY HERE)
# ======================================
# Recommended: set this in Colab Secrets instead of hardcoding
os.environ["OPENAI_API_KEY"] = "sk-REPLACE_WITH_YOUR_KEY"

# ======================================
# OPENAI MODEL CONFIGURATION
# ======================================
# Chosen model: gpt-4o
# Reason: strongest reasoning, stable outputs, regulator-safe language
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.1,     # LOW creativity for nuclear safety
    max_tokens=900
)

# ======================================
# DEFINE NUCLEAR CYBER AGENTS
# ======================================

likelihood_agent = Agent(
    role="Nuclear Cyber Threat Analyst",
    goal="Assess likelihood of cyber compromise in nuclear facilities",
    backstory=(
        "A nuclear cyber security specialist experienced in defense-in-depth, "
        "threat-informed risk assessment, insider threat analysis, and "
        "digital system isolation in accordance with NEI guidance and FANR regulations."
    ),
    llm=llm,
    allow_delegation=False
)

impact_agent = Agent(
    role="Nuclear Safety & Consequence Analyst",
    goal="Evaluate nuclear safety, radiological, and regulatory impact of cyber incidents",
    backstory=(
        "A nuclear safety engineer with deep knowledge of reactor protection systems, "
        "emergency preparedness, safety classification, and FANR licensing basis."
    ),
    llm=llm,
    allow_delegation=False
)

risk_agent = Agent(
    role="Nuclear Cyber Risk Authority",
    goal="Determine overall nuclear cyber risk significance and regulatory acceptability",
    backstory=(
        "A senior nuclear risk authority responsible for determining whether cyber risks "
        "are acceptable, conditionally acceptable, or unacceptable under FANR regulations."
    ),
    llm=llm,
    allow_delegation=False
)

countermeasure_agent = Agent(
    role="Nuclear Cyber Protection Architect",
    goal="Recommend NEI- and FANR-compliant cyber security controls",
    backstory=(
        "A nuclear cyber protection architect specializing in NEI cyber security plans, "
        "defense-in-depth, safety system isolation, monitoring, and regulatory compliance."
    ),
    llm=llm,
    allow_delegation=False
)

# ======================================
# NUCLEAR CYBER RISK ASSESSMENT PIPELINE
# ======================================

def run_nuclear_cyber_risk_assessment(
    facility_type,
    digital_asset,
    safety_class,
    threat_actor,
    network_connectivity,
    defensive_posture,
    vulnerabilities
):
    # --- Likelihood Task ---
    likelihood_task = Task(
        description=f"""
        Perform a nuclear cyber threat likelihood assessment.

        CONSTRAINTS:
        - Use conservative assumptions
        - Do NOT speculate beyond provided inputs
        - Prioritize defense-in-depth
        - Do NOT propose mitigations here

        INPUTS:
        Facility Type: {facility_type}
        Digital Asset: {digital_asset}
        Safety Classification: {safety_class}
        Threat Actor: {threat_actor}
        Network Connectivity: {network_connectivity}
        Defensive Posture: {defensive_posture}
        Known Vulnerabilities: {vulnerabilities}

        OUTPUT FORMAT:
        - Likelihood Rating (Low / Moderate / High)
        - Key Threat Factors
        - NEI Alignment Justification
        """,
        expected_output="Structured likelihood assessment aligned with NEI guidance.",
        agent=likelihood_agent
    )

    # --- Impact Task ---
    impact_task = Task(
        description=f"""
        Assess the nuclear safety and regulatory impact of a cyber compromise.

        INPUTS:
        Digital Asset: {digital_asset}
        Safety Classification: {safety_class}

        CONSIDER:
        - Nuclear safety significance
        - Radiological consequences
        - Emergency preparedness degradation
        - FANR regulatory impact
        - Defense-in-depth degradation

        OUTPUT FORMAT:
        - Impact Level (Low / High / Critical)
        - Affected Safety Functions
        - Regulatory Consequences
        """,
        expected_output="Nuclear impact assessment suitable for regulatory review.",
        agent=impact_agent
    )

    # --- Risk Task ---
    risk_task = Task(
        description="""
        Integrate likelihood and impact assessments.

        RULES:
        - Safety-class systems default to conservative outcomes
        - FANR licensing basis protection takes priority
        - Human safety overrides operational considerations

        OUTPUT FORMAT:
        - Overall Risk Classification (Acceptable / Conditionally Acceptable / Unacceptable)
        - Justification
        - Regulatory Acceptability Statement
        """,
        expected_output="Final nuclear cyber risk determination.",
        agent=risk_agent
    )

    # --- Countermeasure Task ---
    countermeasure_task = Task(
        description="""
        Recommend nuclear-safe cyber security controls.

        REQUIREMENTS:
        - Align with NEI cyber security plans
        - Align with FANR regulations
        - Follow defense-in-depth
        - Avoid operational disruption
        - Prioritize safety systems

        OUTPUT FORMAT:
        - Immediate Controls
        - Medium-Term Controls
        - Governance / Compliance Controls
        """,
        expected_output="Prioritized nuclear cyber security controls.",
        agent=countermeasure_agent
    )

    crew = Crew(
        agents=[
            likelihood_agent,
            impact_agent,
            risk_agent,
            countermeasure_agent
        ],
        tasks=[
            likelihood_task,
            impact_task,
            risk_task,
            countermeasure_task
        ],
        verbose=True
    )

    return crew.kickoff()

# ======================================
# GRADIO UI
# ======================================

def assess_nuclear_cyber_risk(
    facility_type,
    digital_asset,
    safety_class,
    threat_actor,
    network_connectivity,
    defensive_posture,
    vulnerabilities
):
    return run_nuclear_cyber_risk_assessment(
        facility_type,
        digital_asset,
        safety_class,
        threat_actor,
        network_connectivity,
        defensive_posture,
        vulnerabilities
    )

iface = gr.Interface(
    fn=assess_nuclear_cyber_risk,
    inputs=[
        gr.Dropdown(
            ["Nuclear Power Plant", "Research Reactor", "Fuel Handling Facility"],
            label="Facility Type"
        ),
        gr.Textbox(label="Digital Asset", placeholder="Reactor Protection System PLC"),
        gr.Dropdown(
            ["Safety", "Security", "Emergency Preparedness", "Support"],
            label="System Safety Classification"
        ),
        gr.Dropdown(
            ["Nation-State", "Insider", "Terrorist Organization"],
            label="Threat Actor"
        ),
        gr.Dropdown(
            ["Isolated", "Controlled Interface", "Indirect Connectivity"],
            label="Network Connectivity"
        ),
        gr.Dropdown(
            ["Strong", "Moderate", "Weak"],
            label="Defensive Posture"
        ),
        gr.Textbox(
            label="Known Vulnerabilities",
            placeholder="Legacy firmware, shared maintenance laptop"
        )
    ],
    outputs=gr.Textbox(
        lines=35,
        label="Nuclear Cyber Security Risk Assessment Report"
    ),
    title="☢️ Agentic Nuclear Cyber Security Risk Assessment",
    description=(
        "CrewAI-powered nuclear cyber risk assessment using OpenAI reasoning, "
        "aligned with NEI standards and FANR regulatory expectations."
    )
)

# ======================================
# LAUNCH APPLICATION
# ======================================
if __name__ == "__main__":
    iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b625b19d3cd5d2a7ca.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
